# 01_build_cached_inputs_aligned

This notebook rebuilds the cached analysis inputs.

What changed:

- radiology now follows the same preprocessing pattern as pressor and vent
- shared QC shape: hard bounds first, then wide IQR screening
- shared numeric imputation function name: `single_impute_iterative`
- `chiefcomplaint` missingness is kept as `cc_missing` and text is filled with `no_cc`
- `pain_critical` is retained as a structured covariate
- all heavy outputs are written under `./main/shared/...`
- if a cache file already exists, it is loaded and not recomputed

In [7]:
from pathlib import Path
import joblib
import numpy as np
import pandas as pd

from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.decomposition import TruncatedSVD
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import IterativeImputer

ROOT = Path(".")
MAIN = ROOT / "main"
SHARED = MAIN / "shared"
RAD_DIR = SHARED / "radiology"
PRESSOR_DIR = SHARED / "pressor"
VENT_DIR = SHARED / "vent"

for p in [MAIN, SHARED, RAD_DIR, PRESSOR_DIR, VENT_DIR]:
    p.mkdir(parents=True, exist_ok=True)

SEED = 2026
TEXT_COL = "chiefcomplaint"
NO_CC_TOKEN = "no_cc"

RAD_PATH = ROOT / "rad_ed_stay.parquet"
PRESSOR_PATH = ROOT / "analytic_pressor_trigger_MAP_lt_65.parquet"
VENT_PATH = ROOT / "analytic_vent_trigger_SpO2_lt_90.parquet"

RAD_MAX_FOLLOWUP_H = 6.0
RAD_OUTCOME_TIME_COLS = [
    "time_to_any_rad_hours",
    "time_to_advanced_hours",
    "time_to_xray_hours",
]

RAD_NUM_COVS = ["anchor_age", "acuity", "temperature", "heartrate", "resprate", "o2sat", "sbp", "dbp", "pain"]
RAD_CAT_COVS = ["gender", "race", "arrival_transport", "language", "insurance"]

TEXT_Z_DIM = 20
TEXT_MIN_DF = 10
TEXT_MAX_FEATURES = 60000

In [8]:
def _to_str(s: pd.Series) -> pd.Series:
    return s.astype("string").str.strip()

def collapse_race(s: pd.Series) -> pd.Series:
    s = _to_str(s)

    def f(x):
        if pd.isna(x) or x == "":
            return "Unknown"
        u = str(x).upper()

        if "WHITE" in u:
            return "White"
        if "BLACK" in u:
            return "Black"
        if "HISPANIC" in u or "LATINO" in u:
            return "Hispanic/Latino"
        if "ASIAN" in u:
            return "Asian"
        if "PORTUGUESE" in u:
            return "Other"
        if "DECLINED" in u or "UNABLE" in u or "UNKNOWN" in u:
            return "Unknown"
        return "Other"

    return s.map(f)

def collapse_language(s: pd.Series, top_n=8, min_count=1000) -> pd.Series:
    s = _to_str(s)
    s = s.where(~(s.isna() | (s == "")), "Unknown")

    vc = s.value_counts()
    eligible = [x for x in vc.index if (vc[x] >= min_count and x != "Unknown")]
    keep = (["Unknown"] if "Unknown" in vc.index else []) + eligible[:top_n]
    return s.where(s.isin(keep), "Other")

def collapse_insurance(s: pd.Series, min_count=50) -> pd.Series:
    ins = _to_str(s)
    ins = ins.where(~(ins.isna() | (ins == "")), "Unknown")

    vc = ins.value_counts()
    keep = vc.index[vc >= min_count].tolist()
    return ins.where(ins.isin(keep), "Other")

def collapse_simple(s: pd.Series) -> pd.Series:
    x = _to_str(s)
    return x.where(~(x.isna() | (x == "")), "Unknown")

def apply_category_collapsing(df: pd.DataFrame) -> pd.DataFrame:
    d = df.copy()

    if "race" in d.columns:
        d["race"] = collapse_race(d["race"])
    if "language" in d.columns:
        d["language"] = collapse_language(d["language"], top_n=8)
    if "insurance" in d.columns:
        d["insurance"] = collapse_insurance(d["insurance"], min_count=50)
    if "gender" in d.columns:
        d["gender"] = collapse_simple(d["gender"])
    if "arrival_transport" in d.columns:
        d["arrival_transport"] = collapse_simple(d["arrival_transport"])
    if "first_careunit" in d.columns:
        d["first_careunit"] = collapse_simple(d["first_careunit"])
    if "last_careunit" in d.columns:
        d["last_careunit"] = collapse_simple(d["last_careunit"])

    return d

def handle_pain_critical(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    if "pain" not in df.columns:
        df["pain_critical"] = np.int8(0)
        return df

    pain_str = df["pain"].astype("string")
    mask_crit = pain_str.str.strip().str.lower().eq("critical")
    df["pain_critical"] = mask_crit.where(mask_crit.notna(), False).astype("int8")

    pain_num = pd.to_numeric(df["pain"], errors="coerce")
    pain_num = pain_num.where(~mask_crit, 10.0)
    df["pain"] = pain_num
    return df

def fill_chiefcomplaint_missing(df: pd.DataFrame, text_col: str = TEXT_COL, fill_value: str = NO_CC_TOKEN) -> pd.DataFrame:
    df = df.copy()

    raw = df[text_col].astype("string")
    stripped = raw.str.strip()
    missing = stripped.isna() | stripped.eq("")

    df["cc_missing"] = missing.astype("int8")
    df[text_col] = stripped.where(~missing, fill_value)
    return df

def add_radiology_followup(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    df["trigger_time"] = pd.to_datetime(df["trigger_time"])
    df["ed_outtime"] = pd.to_datetime(df["ed_outtime"])

    out_cap = df["trigger_time"] + pd.to_timedelta(RAD_MAX_FOLLOWUP_H, unit="h")
    df["outtime"] = df["ed_outtime"].where(df["ed_outtime"] < out_cap, out_cap)
    df["follow_up_hours"] = (df["outtime"] - df["trigger_time"]) / np.timedelta64(1, "h")
    return df

def minimal_nonsense_qc(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    rad_rules = {
        "o2sat": (0.0, 100.0),
        "sbp": (1.0, 500.0),
        "dbp": (1.0, 400.0),
        "heartrate": (0.0, 500.0),
        "resprate": (0.0, 300.0),
        "temperature": (10.0, 200.0),
        "pain": (0.0, 10.0),
        "anchor_age": (0.0, 120.0),
    }

    if "temperature" in df.columns:
        t = pd.to_numeric(df["temperature"], errors="coerce")

        mask_c_like = t.notna() & (t >= 25.0) & (t <= 45.0)
        df["temperature_c_like"] = mask_c_like.astype("int8")
        df.loc[mask_c_like, "temperature"] = np.nan

        t2 = pd.to_numeric(df["temperature"], errors="coerce")
        bad_f = t2.notna() & ((t2 < 80.0) | (t2 > 110.0))
        df.loc[bad_f, "temperature"] = np.nan
    else:
        df["temperature_c_like"] = np.int8(0)

    for col, (lo, hi) in rad_rules.items():
        if col == "temperature":
            continue
        if col not in df.columns:
            continue
        x = pd.to_numeric(df[col], errors="coerce")
        bad = x.notna() & ((x < lo) | (x > hi))
        df.loc[bad, col] = np.nan

    return df

In [9]:
def radiology_summary_qc(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    numeric_cols = [c for c in RAD_NUM_COVS if c in df.columns]
    if "anchor_age_sq" in df.columns:
        numeric_cols.append("anchor_age_sq")

    hard_bounds = {
        "anchor_age": (0.0, 120.0),
        "anchor_age_sq": (0.0, 120.0 ** 2),
        "acuity": (1.0, 5.0),
        "temperature": (80.0, 110.0),
        "heartrate": (10.0, 300.0),
        "resprate": (1.0, 80.0),
        "o2sat": (20.0, 100.0),
        "sbp": (20.0, 300.0),
        "dbp": (5.0, 250.0),
        "pain": (0.0, 10.0),
    }

    for c in numeric_cols:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    for c, (lo, hi) in hard_bounds.items():
        if c in df.columns:
            s = pd.to_numeric(df[c], errors="coerce")
            df[c] = s.where((s >= lo) & (s <= hi))

    for c in numeric_cols:
        s = pd.to_numeric(df[c], errors="coerce")
        vals = s.dropna()
        if len(vals) < 20:
            continue

        q1 = vals.quantile(0.25)
        q3 = vals.quantile(0.75)
        iqr = q3 - q1
        if not np.isfinite(iqr) or iqr <= 0:
            continue

        lo = q1 - 10.0 * iqr
        hi = q3 + 10.0 * iqr
        df[c] = s.where((s >= lo) & (s <= hi))

    return df

def icu_summary_qc(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    hard_bounds = {
        "temp_last": (25.0, 45.0),
        "hr_last": (10.0, 300.0),
        "rr_last": (1.0, 80.0),
        "spo2_last": (20.0, 100.0),
        "sbp_last": (20.0, 300.0),
        "dbp_last": (5.0, 250.0),
        "map_last": (20.0, 250.0),
        "lactate_last": (0.0, 30.0),
        "ph_last": (6.8, 7.8),
        "pco2_last": (5.0, 200.0),
        "po2_last": (10.0, 500.0),
        "glucose_last": (10.0, 1500.0),
        "sodium_last": (100.0, 200.0),
        "potassium_last": (1.5, 10.0),
        "chloride_last": (50.0, 150.0),
        "bicarbonate_last": (1.0, 60.0),
        "creatinine_last": (0.1, 25.0),
        "bun_last": (1.0, 250.0),
        "hemoglobin_last": (1.0, 25.0),
        "wbc_last": (0.1, 300.0),
        "platelet_last": (1.0, 2000.0),
        "bilirubin_last": (0.0, 80.0),
        "ast_last": (0.0, 10000.0),
        "alt_last": (0.0, 10000.0),
    }

    last_cols = [c for c in df.columns if c.endswith("_last")]
    for c in last_cols:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    if "trigger_value" in df.columns:
        df["trigger_value"] = pd.to_numeric(df["trigger_value"], errors="coerce")
    if "time_since_icu_admission_hours" in df.columns:
        df["time_since_icu_admission_hours"] = pd.to_numeric(df["time_since_icu_admission_hours"], errors="coerce")

    for c, (lo, hi) in hard_bounds.items():
        if c in df.columns:
            s = df[c]
            df[c] = s.where((s >= lo) & (s <= hi))

    if "trigger_value" in df.columns:
        if "map_last" in df.columns:
            df["trigger_value"] = df["trigger_value"].where(
                (df["trigger_value"] >= 20.0) & (df["trigger_value"] <= 250.0)
            )
        elif "spo2_last" in df.columns:
            df["trigger_value"] = df["trigger_value"].where(
                (df["trigger_value"] >= 20.0) & (df["trigger_value"] <= 100.0)
            )

    if "time_since_icu_admission_hours" in df.columns:
        df["time_since_icu_admission_hours"] = df["time_since_icu_admission_hours"].where(
            (df["time_since_icu_admission_hours"] >= 0.0) &
            (df["time_since_icu_admission_hours"] <= 24.0 * 30.0)
        )

    for c in last_cols:
        s = pd.to_numeric(df[c], errors="coerce")
        vals = s.dropna()
        if len(vals) < 20:
            continue

        q1 = vals.quantile(0.25)
        q3 = vals.quantile(0.75)
        iqr = q3 - q1
        if not np.isfinite(iqr) or iqr <= 0:
            continue

        lo = q1 - 10.0 * iqr
        hi = q3 + 10.0 * iqr
        df[c] = s.where((s >= lo) & (s <= hi))

    return df

In [10]:
def single_impute_iterative(
    df: pd.DataFrame,
    num_covs: list[str],
    cat_covs: list[str],
    seed: int,
    max_iter: int = 50,
    tol: float = 1e-2,
    n_nearest_features: int | None = 25,
) -> pd.DataFrame:
    df = df.copy()

    num_covs = [c for c in num_covs if c in df.columns]
    cat_covs = [c for c in cat_covs if c in df.columns]

    X = df[num_covs].apply(pd.to_numeric, errors="coerce")

    bad_cols = [c for c in num_covs if X[c].notna().sum() < 2]
    assert len(bad_cols) == 0, f"Columns unusable for iterative imputation: {bad_cols}"

    for c in num_covs:
        df[f"{c}_missing"] = X[c].isna().astype("int8")

    lower = X.min(skipna=True)
    upper = X.max(skipna=True)

    nunique_obs = X.nunique(dropna=True)
    near_constant_cols = [c for c in num_covs if nunique_obs[c] <= 1]

    impute_cols = [c for c in num_covs if c not in near_constant_cols]

    if len(impute_cols) > 0:
        X_imp_in = X[impute_cols].copy()

        min_value = lower[impute_cols].to_numpy(dtype=float)
        max_value = upper[impute_cols].to_numpy(dtype=float)

        if n_nearest_features is not None:
            n_nearest_features = min(n_nearest_features, max(1, len(impute_cols) - 1))
        if len(impute_cols) <= 2:
            n_nearest_features = None

        imp = IterativeImputer(
            estimator=ExtraTreesRegressor(
                n_estimators=100,
                max_depth=None,
                min_samples_leaf=5,
                n_jobs=-1,
                random_state=seed,
            ),
            max_iter=max_iter,
            tol=tol,
            random_state=seed,
            initial_strategy="median",
            imputation_order="ascending",
            skip_complete=True,
            n_nearest_features=n_nearest_features,
            min_value=min_value,
            max_value=max_value,
            sample_posterior=False,
        )

        X_imp = imp.fit_transform(X_imp_in)
        X_imp = pd.DataFrame(X_imp, columns=impute_cols, index=df.index)
        X_imp = X_imp.clip(lower=lower[impute_cols], upper=upper[impute_cols], axis=1)

        assert np.isfinite(X_imp.to_numpy(dtype=float)).all(), "Non-finite values after iterative imputation."

        df[impute_cols] = X_imp

    for c in near_constant_cols:
        observed = X[c].dropna()
        assert len(observed) > 0, f"Column {c} has no observed values."
        df[c] = X[c].where(X[c].notna(), observed.iloc[0])

    for c in cat_covs:
        if pd.api.types.is_numeric_dtype(df[c]):
            continue
        s = df[c].astype("string")
        missing = s.isna() | s.str.strip().eq("")
        df[c] = s.where(~missing, "Unknown")

    return df

def fit_text_assets(df: pd.DataFrame, text_col: str, out_dir: Path, prefix: str):
    vec_path = out_dir / f"{prefix}_tfidf.joblib"
    svd_path = out_dir / f"{prefix}_svd.joblib"
    z_path = out_dir / f"{prefix}_text_z.parquet"

    if vec_path.exists() and svd_path.exists() and z_path.exists():
        vec = joblib.load(vec_path)
        svd = joblib.load(svd_path)
        z_df = pd.read_parquet(z_path)
        return vec, svd, z_df

    txt_s = df[text_col].astype("string").str.strip()
    txt = txt_s.where(txt_s.notna() & txt_s.ne(""), NO_CC_TOKEN).astype(str).to_numpy()

    vec = TfidfVectorizer(
        ngram_range=(1, 2),
        min_df=TEXT_MIN_DF,
        max_features=TEXT_MAX_FEATURES,
        lowercase=True,
        dtype=np.float64,
    )
    X_tfidf = vec.fit_transform(txt)

    assert np.isfinite(X_tfidf.data).all(), "Non-finite values found in TF-IDF matrix."

    max_rank = min(X_tfidf.shape[0] - 1, X_tfidf.shape[1] - 1)
    n_components = min(TEXT_Z_DIM, max_rank)
    assert n_components >= 1, "TF-IDF matrix rank is too small for TruncatedSVD."

    svd = TruncatedSVD(
        n_components=n_components,
        algorithm="arpack",
        random_state=SEED,
    )
    Z = svd.fit_transform(X_tfidf)

    assert np.isfinite(Z).all(), "Non-finite values found in SVD output."

    z_cols = [f"z_{i:02d}" for i in range(Z.shape[1])]
    z_df = pd.DataFrame(Z, columns=z_cols, index=df.index)

    joblib.dump(vec, vec_path)
    joblib.dump(svd, svd_path)
    z_df.to_parquet(z_path, index=False)

    return vec, svd, z_df

In [11]:
def validate_imputation_output(df: pd.DataFrame, num_cols: list[str], name: str):
    X = df[num_cols].to_numpy(dtype=float)

    has_nan = np.isnan(X).any()
    has_inf = np.isinf(X).any()

    if has_nan or has_inf:
        bad_mask = ~np.isfinite(X)
        bad_cols = np.array(num_cols)[bad_mask.any(axis=0)]

        print(f"\n[ERROR] Non-finite values detected after imputation in {name}")
        print("Columns:", bad_cols.tolist())

        raise ValueError("Imputation produced NaN or inf values.")

    print(f"[OK] Imputation output clean for {name}")

## Radiology input cache

In [12]:
def build_radiology_cached_input(src_path: Path, out_dir: Path, prefix: str):
    cache_path = out_dir / f"{prefix}_analytic_imputed.parquet"

    if cache_path.exists():
        analytic = pd.read_parquet(cache_path)
    else:
        analytic = pd.read_parquet(src_path)

        analytic = add_radiology_followup(analytic)
        analytic = handle_pain_critical(analytic)
        analytic = apply_category_collapsing(analytic)
        analytic = minimal_nonsense_qc(analytic)

        drop_summary_cols = [c for c in analytic.columns if c.endswith(("_mean", "_min", "_max"))]
        if drop_summary_cols:
            analytic = analytic.drop(columns=drop_summary_cols)

        analytic = radiology_summary_qc(analytic)
        analytic = fill_chiefcomplaint_missing(analytic, TEXT_COL, NO_CC_TOKEN)

        if "anchor_age" in analytic.columns:
            analytic["anchor_age"] = pd.to_numeric(analytic["anchor_age"], errors="coerce")
            analytic["anchor_age_sq"] = analytic["anchor_age"] ** 2

        num_cols = [c for c in analytic.columns if c.endswith("_last")]
        for c in ["anchor_age", "anchor_age_sq", "trigger_value", "time_since_icu_admission_hours"]:
            if c in analytic.columns:
                num_cols.append(c)
        num_cols = sorted(set(num_cols))

        cat_cols = [
            c
            for c in [
                "gender",
                "race",
                "language",
                "insurance",
                "first_careunit",
                "last_careunit",
            ]
            if c in analytic.columns
        ]

        print(f"\n[{prefix}] numeric columns used for imputation ({len(num_cols)}):")
        for c in num_cols:
            print(c)

        analytic = single_impute_iterative(
            analytic,
            num_covs=num_cols,
            cat_covs=cat_cols,
            seed=SEED,
            max_iter=20,
            tol=1e-3,
            n_nearest_features=25,
        )
        validate_imputation_output(analytic, num_cols, prefix)

        analytic.to_parquet(cache_path, index=False)

    _, _, z_df = fit_text_assets(
        df=analytic,
        text_col=TEXT_COL,
        out_dir=out_dir,
        prefix=prefix,
    )

    bundle_path = out_dir / f"{prefix}_text_bundle.parquet"
    if not bundle_path.exists():
        id_cols = [c for c in ["subject_id", "hadm_id", "ed_stay_id", TEXT_COL, "cc_missing"] if c in analytic.columns]
        pd.concat(
            [analytic[id_cols].reset_index(drop=True), z_df.reset_index(drop=True)],
            axis=1,
        ).to_parquet(bundle_path, index=False)

    return analytic, z_df

radiology_analytic, radiology_z = build_radiology_cached_input(RAD_PATH, RAD_DIR, "radiology")

print("radiology:", radiology_analytic.shape, radiology_z.shape)


[radiology] numeric columns used for imputation (16):
anchor_age
anchor_age_sq
creatinine_last
dbp_last
hr_last
lactate_last
map_last
pain_last
platelets_last
potassium_last
rr_last
sbp_last
sodium_last
spo2_last
temp_last
wbc_last


/Users/lauzhenyi/Library/Python/3.11/lib/python/site-packages/sklearn/impute/_iterative.py:825: ConvergenceWarning: [IterativeImputer] Early stopping criterion not reached.
  warnings.warn(


[OK] Imputation output clean for radiology
radiology: (203016, 122) (203016, 20)


## ICU pressor / vent input caches

In [13]:
def build_icu_cached_input(src_path: Path, out_dir: Path, prefix: str):
    cache_path = out_dir / f"{prefix}_analytic_imputed.parquet"

    if cache_path.exists():
        analytic = pd.read_parquet(cache_path)
    else:
        analytic = pd.read_parquet(src_path)

        analytic = handle_pain_critical(analytic)
        analytic = apply_category_collapsing(analytic)
        analytic = minimal_nonsense_qc(analytic)

        drop_summary_cols = [c for c in analytic.columns if c.endswith(("_mean", "_min", "_max"))]
        if drop_summary_cols:
            analytic = analytic.drop(columns=drop_summary_cols)

        analytic = icu_summary_qc(analytic)
        analytic = fill_chiefcomplaint_missing(analytic, TEXT_COL, NO_CC_TOKEN)

        if "anchor_age" in analytic.columns:
            analytic["anchor_age"] = pd.to_numeric(analytic["anchor_age"], errors="coerce")
            analytic["anchor_age_sq"] = analytic["anchor_age"] ** 2

        num_cols = [c for c in analytic.columns if c.endswith("_last")]
        for c in ["anchor_age", "anchor_age_sq", "trigger_value", "time_since_icu_admission_hours"]:
            if c in analytic.columns:
                num_cols.append(c)
        num_cols = sorted(set(num_cols))

        cat_cols = [
            c
            for c in [
                "gender",
                "race",
                "language",
                "insurance",
                "first_careunit",
                "last_careunit",
            ]
            if c in analytic.columns
        ]

        print(f"\n[{prefix}] numeric columns used for imputation ({len(num_cols)}):")
        for c in num_cols:
            print(c)

        analytic = single_impute_iterative(
            analytic,
            num_covs=num_cols,
            cat_covs=cat_cols,
            seed=SEED,
            max_iter=20,
            tol=1e-3,
            n_nearest_features=25,
        )
        validate_imputation_output(analytic, num_cols, prefix)

        analytic.to_parquet(cache_path, index=False)

    _, _, z_df = fit_text_assets(
        df=analytic,
        text_col=TEXT_COL,
        out_dir=out_dir,
        prefix=prefix,
    )

    bundle_path = out_dir / f"{prefix}_text_bundle.parquet"
    if not bundle_path.exists():
        id_cols = [c for c in ["subject_id", "hadm_id", "stay_id", TEXT_COL, "cc_missing"] if c in analytic.columns]
        pd.concat(
            [analytic[id_cols].reset_index(drop=True), z_df.reset_index(drop=True)],
            axis=1,
        ).to_parquet(bundle_path, index=False)

    return analytic, z_df

pressor_analytic, pressor_z = build_icu_cached_input(PRESSOR_PATH, PRESSOR_DIR, "pressor")
vent_analytic, vent_z = build_icu_cached_input(VENT_PATH, VENT_DIR, "vent")

print("pressor:", pressor_analytic.shape, pressor_z.shape)
print("vent:", vent_analytic.shape, vent_z.shape)


[pressor] numeric columns used for imputation (17):
anchor_age
anchor_age_sq
creatinine_last
dbp_last
hr_last
lactate_last
map_last
platelets_last
potassium_last
rr_last
sbp_last
sodium_last
spo2_last
temp_last
time_since_icu_admission_hours
trigger_value
wbc_last


/Users/lauzhenyi/Library/Python/3.11/lib/python/site-packages/sklearn/impute/_iterative.py:825: ConvergenceWarning: [IterativeImputer] Early stopping criterion not reached.
  warnings.warn(


[OK] Imputation output clean for pressor

[vent] numeric columns used for imputation (17):
anchor_age
anchor_age_sq
creatinine_last
dbp_last
hr_last
lactate_last
map_last
platelets_last
potassium_last
rr_last
sbp_last
sodium_last
spo2_last
temp_last
time_since_icu_admission_hours
trigger_value
wbc_last


/Users/lauzhenyi/Library/Python/3.11/lib/python/site-packages/sklearn/impute/_iterative.py:825: ConvergenceWarning: [IterativeImputer] Early stopping criterion not reached.
  warnings.warn(


[OK] Imputation output clean for vent
pressor: (69223, 66) (69223, 20)
vent: (34042, 66) (34042, 20)


## Cache manifest

In [14]:
manifest = pd.DataFrame(
    {
        "path": sorted(str(p.relative_to(ROOT)) for p in SHARED.rglob("*") if p.is_file())
    }
)
manifest_path = SHARED / "manifest.csv"
manifest.to_csv(manifest_path, index=False)
manifest

,path
0,main/shared/pressor/pressor_analytic_imputed.p...
1,main/shared/pressor/pressor_svd.joblib
2,main/shared/pressor/pressor_text_bundle.parquet
3,main/shared/pressor/pressor_text_z.parquet
4,main/shared/pressor/pressor_tfidf.joblib
5,main/shared/radiology/radiology_analytic_imput...
6,main/shared/radiology/radiology_svd.joblib
7,main/shared/radiology/radiology_text_bundle.pa...
8,main/shared/radiology/radiology_text_z.parquet
9,main/shared/radiology/radiology_tfidf.joblib
